# Figure 1: Repertoire size, TCR diversity, and average clonality as a function of age

This notebook reproduces Figure 1 of the manuscript.

The figure shows age-associated trends in:

- repertoire size, \(S\), defined as the total number of productive TCRs sequenced,
- TCR diversity, \(D\), defined as the number of unique productive TCR rearrangements,
- average clonality, \(S/D\), defined as the mean number of productive T cells per unique rearrangement.

For each quantity, we plot the median in decade-wide age bins, bootstrap uncertainties on the median, and the central 50% and 90% of subjects in each bin.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import interp1d

# Render figures cleanly in notebooks.
%config InlineBackend.figure_format = "pdf"

# Global plotting defaults.
plt.rcParams["xtick.labelsize"] = 14
plt.rcParams["ytick.labelsize"] = 14
plt.rcParams["axes.labelsize"] = 16

# Use Matplotlib default color cycle for consistency with the original manuscript figures.
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# Set a random seed so bootstrap error bars are reproducible.
rng = np.random.default_rng(1)

In [2]:
# Public data table hosted on Zenodo.
# This table contains subject-level TCR repertoire metrics used in the manuscript.
data_url = "https://zenodo.org/records/13993996/files/TCR_size_diversity_age_sex.csv?download=1"

df = (
    pd.read_csv(data_url)
    .rename(
        columns={
            "#Subject_ID": "subject_id",
            " age": "age",
            " sex": "sex",
            " size": "repertoire_size",
            " diversity": "tcr_diversity",
        }
    )
)

# Remove leading/trailing whitespace from sex labels.
df["sex"] = df["sex"].str.strip()

# Add log-transformed quantities used for Figure 1.
df["log_repertoire_size"] = np.log10(df["repertoire_size"])
df["log_tcr_diversity"] = np.log10(df["tcr_diversity"])
df["log_average_clonality"] = np.log10(df["repertoire_size"] / df["tcr_diversity"])

df.head()

,subject_id,age,sex,repertoire_size,tcr_diversity,log_repertoire_size,log_tcr_diversity,log_average_clonality
0,30000,63,male,508139,274815,5.705983,5.439040,0.266942
1,30001,76,female,263716,167611,5.421136,5.224303,0.196834
2,30002,70,female,519416,261836,5.715515,5.418029,0.297486
3,30003,44,male,576294,383460,5.760644,5.583720,0.176924
4,30004,47,male,505447,290642,5.703676,5.463358,0.240317


In [3]:
def bootstrap_statistic(values, statistic=np.median, n_bootstrap=1001, rng=None):
    """
    Estimate the bootstrap uncertainty of a statistic.

    Parameters
    ----------
    values : array-like
        Values to bootstrap.
    statistic : callable
        Statistic to calculate. Default is np.median.
    n_bootstrap : int
        Number of bootstrap realizations.
    rng : np.random.Generator
        Random number generator for reproducibility.

    Returns
    -------
    point_estimate : float
        Statistic evaluated on the original data.
    bootstrap_error : float
        Standard deviation of bootstrap statistics.
    """
    if rng is None:
        rng = np.random.default_rng()

    values = np.asarray(values)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan

    bootstrap_values = []
    for _ in range(n_bootstrap):
        sample = rng.choice(values, size=len(values), replace=True)
        bootstrap_values.append(statistic(sample))

    return statistic(values), np.std(bootstrap_values)


def binned_summary(x, y, bins, interval=(0.25, 0.75), n_bootstrap=1001, rng=None):
    """
    Calculate binned medians, bootstrap errors on the median,
    and central intervals of y in bins of x.

    Parameters
    ----------
    x : array-like
        Binning variable, here subject age.
    y : array-like
        Quantity to summarize.
    bins : list of tuple
        List of (lower, upper) bin edges.
    interval : tuple
        Lower and upper quantiles for the central interval.
        For example, (0.25, 0.75) gives the central 50%.
    n_bootstrap : int
        Number of bootstrap realizations for median uncertainty.
    rng : np.random.Generator
        Random number generator for reproducibility.

    Returns
    -------
    summary : pd.DataFrame
        One row per age bin with median x, median y, bootstrap error,
        and central interval bounds.
    """
    if rng is None:
        rng = np.random.default_rng()

    x = np.asarray(x)
    y = np.asarray(y)

    rows = []

    for lower, upper in bins:
        in_bin = (
            np.isfinite(x)
            & np.isfinite(y)
            & (x >= lower)
            & (x < upper)
        )

        x_bin = x[in_bin]
        y_bin = y[in_bin]

        if len(y_bin) <= 1:
            continue

        y_median, y_error = bootstrap_statistic(
            y_bin,
            statistic=np.median,
            n_bootstrap=n_bootstrap,
            rng=rng,
        )

        rows.append(
            {
                "age_median": np.median(x_bin),
                "value_median": y_median,
                "value_error": y_error,
                "interval_low": np.quantile(y_bin, interval[0]),
                "interval_high": np.quantile(y_bin, interval[1]),
                "n_subjects": len(y_bin),
            }
        )

    return pd.DataFrame(rows)

In [4]:
# Decade-wide bins from 0--10 through 80--100.
# The final bin is extended to 100 to include the oldest subjects.
age_bin_lower = np.arange(9) * 10
age_bin_upper = age_bin_lower + 10
age_bin_upper[-1] = 100

age_bins = list(zip(age_bin_lower, age_bin_upper))
age_bins

[(np.int64(0), np.int64(10)),
 (np.int64(10), np.int64(20)),
 (np.int64(20), np.int64(30)),
 (np.int64(30), np.int64(40)),
 (np.int64(40), np.int64(50)),
 (np.int64(50), np.int64(60)),
 (np.int64(60), np.int64(70)),
 (np.int64(70), np.int64(80)),
 (np.int64(80), np.int64(100))]

In [5]:
# Central intervals shown in Figure 1.
central_50 = (0.25, 0.75)
central_90 = (0.05, 0.95)

# Number of bootstrap realizations.
# The manuscript used 10,001 for final values; 1,001 is faster and visually indistinguishable.
# Set to 10001 if you want exact manuscript-level bootstrap precision.
n_bootstrap = 1001

fig1_s_50 = binned_summary(
    df["age"],
    df["log_repertoire_size"],
    bins=age_bins,
    interval=central_50,
    n_bootstrap=n_bootstrap,
    rng=rng,
)

fig1_s_90 = binned_summary(
    df["age"],
    df["log_repertoire_size"],
    bins=age_bins,
    interval=central_90,
    n_bootstrap=n_bootstrap,
    rng=rng,
)

fig1_d_50 = binned_summary(
    df["age"],
    df["log_tcr_diversity"],
    bins=age_bins,
    interval=central_50,
    n_bootstrap=n_bootstrap,
    rng=rng,
)

fig1_d_90 = binned_summary(
    df["age"],
    df["log_tcr_diversity"],
    bins=age_bins,
    interval=central_90,
    n_bootstrap=n_bootstrap,
    rng=rng,
)

fig1_clonality_50 = binned_summary(
    df["age"],
    df["log_average_clonality"],
    bins=age_bins,
    interval=central_50,
    n_bootstrap=n_bootstrap,
    rng=rng,
)

fig1_clonality_90 = binned_summary(
    df["age"],
    df["log_average_clonality"],
    bins=age_bins,
    interval=central_90,
    n_bootstrap=n_bootstrap,
    rng=rng,
)

fig1_d_50

,age_median,value_median,value_error,interval_low,interval_high,n_subjects
0,18.0,5.622811,0.007743,5.509808,5.709925,561
1,25.0,5.591481,0.004662,5.496075,5.678771,1591
2,36.0,5.553383,0.002556,5.456033,5.641961,5043
3,45.0,5.518959,0.002373,5.412363,5.608608,7608
4,54.0,5.495780,0.002569,5.385761,5.596891,8252
5,63.0,5.449432,0.002798,5.328338,5.555464,5491
6,73.0,5.377487,0.004642,5.242681,5.497548,1647
7,82.0,5.273809,0.024268,5.105241,5.425069,237


In [6]:
def plot_panel(ax, summary_50, summary_90, ylabel, panel_label, legend_loc="lower left"):
    """
    Plot one Figure 1 panel.
    """
    ax.fill_between(
        summary_90["age_median"],
        summary_90["interval_low"],
        summary_90["interval_high"],
        alpha=0.5,
        label="Central 90%",
        color=colors[1],
    )

    ax.fill_between(
        summary_50["age_median"],
        summary_50["interval_low"],
        summary_50["interval_high"],
        alpha=0.5,
        label="Central 50%",
        color=colors[0],
    )

    ax.errorbar(
        summary_50["age_median"],
        summary_50["value_median"],
        yerr=summary_50["value_error"],
        label="Median",
        linewidth=3,
    )

    ax.set_xlabel("Age")
    ax.set_ylabel(ylabel)
    ax.legend(loc=legend_loc, fontsize=12)

    # Place panel labels in axes coordinates so they are robust to axis limits.
    ax.text(
        0.93,
        0.93,
        panel_label,
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        ha="right",
        va="top",
    )


fig, axes = plt.subplots(1, 3, figsize=(20, 6))

plot_panel(
    axes[0],
    fig1_s_50,
    fig1_s_90,
    ylabel="Repertoire Size (Log $S$)",
    panel_label="(A)",
    legend_loc="lower left",
)

plot_panel(
    axes[1],
    fig1_d_50,
    fig1_d_90,
    ylabel="Repertoire Diversity (Log $D$)",
    panel_label="(B)",
    legend_loc="lower left",
)

plot_panel(
    axes[2],
    fig1_clonality_50,
    fig1_clonality_90,
    ylabel="Average Clonality (Log $S/D$)",
    panel_label="(C)",
    legend_loc="upper left",
)

axes[2].set_ylim(0.05, 0.66)

fig.tight_layout()

# Uncomment to save the figure.
# fig.savefig("fig1.pdf", bbox_inches="tight")

plt.show()

<Figure size 2000x600 with 3 Axes>

In [7]:
# The manuscript quotes the peak-to-peak intrinsic biological scatter in TCR diversity
# for the central 90% of subjects after accounting for measurement uncertainty.
#
# Measurement uncertainty in Log D from repeat samples is approximately 0.10 dex.
# For the central 90%, the corresponding half-width is 1.645 * sigma.
#
# For each age bin:
# observed_half_width = (central_90_high - central_90_low) / 2
# intrinsic_half_width = sqrt(observed_half_width^2 - (1.645 * measurement_sigma)^2)
# peak_to_peak_intrinsic_width = 2 * intrinsic_half_width
#
# We convert dex widths into multiplicative factors using 10^dex.

measurement_sigma_log_d = 0.10
central_90_sigma_factor = 1.645

observed_half_width = (
    fig1_d_90["interval_high"] - fig1_d_90["interval_low"]
) / 2

intrinsic_half_width = np.sqrt(
    observed_half_width**2
    - (central_90_sigma_factor * measurement_sigma_log_d) ** 2
)

peak_to_peak_intrinsic_width_dex = 2 * intrinsic_half_width
peak_to_peak_intrinsic_factor = 10 ** peak_to_peak_intrinsic_width_dex

intrinsic_scatter_summary = pd.DataFrame(
    {
        "age_median": fig1_d_90["age_median"],
        "observed_central_90_width_dex": 2 * observed_half_width,
        "intrinsic_central_90_width_dex": peak_to_peak_intrinsic_width_dex,
        "intrinsic_central_90_factor": peak_to_peak_intrinsic_factor,
    }
)

intrinsic_scatter_summary

,age_median,observed_central_90_width_dex,intrinsic_central_90_width_dex,intrinsic_central_90_factor
0,18.0,0.470344,0.336128,2.168345
1,25.0,0.473261,0.340198,2.188759
2,36.0,0.472173,0.338684,2.181141
3,45.0,0.492962,0.367111,2.328686
4,54.0,0.527697,0.412582,2.585721
5,63.0,0.594156,0.494753,3.124300
6,73.0,0.670894,0.584686,3.843140
7,82.0,0.773187,0.699698,5.008389
